## Incremental Backup in Weaviate

Weaviate version 1.36.6+ has introduced incremental backups!

Here is how to use it.

Notes:
- If using multi node, Target cluster must have same node count as source
- Collections being restored must not exist on target
- Node names must match (unless using node mapping)

Production Recommendations:

- Use incremental backups for regular schedules to reduce storage and transfer costs
- Schedule full backups periodically (e.g., weekly) with incremental backups in between
- Monitor backup sizes - incremental backups should be smaller than full backups suite.go:654-657
- Test restoration procedures regularly
- Use compression to optimize storage and transfer

In [ ]:
!pip install weaviate-client numpy

In [1]:
# lets create an embedded server

import os, shutil
import weaviate

backup_path = "/tmp/weaviate-backup"
if not os.path.exists(backup_path):
    os.makedirs(backup_path)
else:
    shutil.rmtree(backup_path)
    os.makedirs(backup_path)

client = weaviate.connect_to_embedded(
    version="1.36.6",
    environment_variables={
        "ENABLE_MODULES": "backup-filesystem",
        "BACKUP_FILESYSTEM_PATH": backup_path,
    },
)

{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-04-06T16:43:57-03:00"}
{"action":"startup","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-04-06T16:43:57-03:00"}
{"action":"startup","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-04-06T16:43:57-03:00"}
{"action":"startup","auto_schema_enabled":true,"build_git_commit":"5bbdf290fd","build_go_version

{"action":"startup","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"Configuring crons","time":"2026-04-06T16:44:16-03:00"}
{"action":"cron","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","job":"trigger_objects_ttl_deletion","level":"info","msg":"cron job skipped, no schedule","time":"2026-04-06T16:44:16-03:00"}


In [2]:
import os

# HELPERS

def list_backup_files(recursive=False):
    if recursive:
        backup_files = []
        for root, dirs, files in os.walk("/tmp/weaviate-backup"):
            for file in files:
                backup_files.append(os.path.join(root, file))
        return backup_files
    else:
        return os.listdir("/tmp/weaviate-backup")

def humanize_size(total_size):
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if total_size < 1024:
            break
        total_size /= 1024
    return f"{total_size:.2f} {unit}"

def get_folder_size_bytes(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total

def get_backup_folder_size(subfolder=None):
    backup_path = "/tmp/weaviate-backup"
    if subfolder:
        return get_folder_size_bytes(os.path.join(backup_path, subfolder))

    rows = []
    for sub in sorted(os.listdir(backup_path)):
        sub_path = os.path.join(backup_path, sub)
        if os.path.isdir(sub_path):
            rows.append((sub, get_folder_size_bytes(sub_path)))

    if not rows:
        return "No backups found."

    col_width = max(len(name) for name, _ in rows)
    header = f"{'Backup ID':<{col_width}}  {'Size':>10}"
    separator = "-" * len(header)
    lines = [header, separator]
    for name, size in rows:
        lines.append(f"{name:<{col_width}}  {humanize_size(size):>10}")
    lines.append(separator)
    lines.append(f"{'TOTAL':<{col_width}}  {humanize_size(sum(s for _, s in rows)):>10}")
    print("\n".join(lines))

In [3]:
list_backup_files()

[]

# Let's ingest some content

In [4]:
import numpy as np

# let's create our base content
client.collections.delete("IncrementalBackupCollection")

collection = client.collections.create("IncrementalBackupCollection")

# let's add some random content to generate some data to backup
with collection.batch.fixed_size(100) as batch:
    for i in range(10000):
        # generate 1000 words random content
        random_content = " ".join([f"word{j}" for j in range(1000)])
        batch.add_object(
            {"name": f"Object {i}", "content": random_content},
            vector=np.random.rand(1536).tolist(),
        )


{"action":"restore_from_disk","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","class":"IncrementalBackupCollection","level":"info","msg":"no snapshot found, loading from commit log","shard":"Bu0eQHd9E617","targetVector":"default","time":"2026-04-06T16:44:27-03:00"}
{"action":"restore_from_disk","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","class":"IncrementalBackupCollection","duration":"389.833µs","level":"info","msg":"restored data from disk","shard":"Bu0eQHd9E617","targetVector":"default","time":"2026-04-06T16:44:27-03:00"}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"Created shard incrementalbackupcollection_Bu0eQHd9E617 in 3.702333ms","time":"2026-04-06T16:44:27-03:00"}
{"action":"lazy_shard_auto_detection","build_git_commit":"5bbdf290fd","build_go_versio

In [5]:
collection.aggregate.over_all().total_count

10000

# We have now 10k objects

In [6]:
# this is an example of object we created
example_object = collection.query.fetch_objects(include_vector=True).objects[0]
print("Example object:", example_object.properties["name"], ",", example_object.properties["content"][0:30], "...")
print("Vector:", example_object.vector["default"][0:5], "...")

Example object: Object 2121 , word0 word1 word2 word3 word4  ...
Vector: [0.5514935255050659, 0.18769460916519165, 0.8909557461738586, 0.6026855707168579, 0.35704848170280457] ...


now let's create a base backup. This base backup can be used as reference for next incremental backups

In [7]:
client.backup.create(
    backup_id="base_backup01-01-01",
    backend="filesystem", # if you are using s3, or gcs, change it here and provide credentials as environment variables
    wait_for_completion=True,
)

{"action":"try_backup","backend":"filesystem","backup_id":"base_backup01-01-01","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:44:48-03:00","took":1445958}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","hardlinks_supported":true,"level":"info","msg":"backup: probed filesystem hardlink support","time":"2026-04-06T16:44:48-03:00"}
{"action":"backup_status","backend":"filesystem","backup_id":"base_backup01-01-01","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:44:48-03:00","took":3500}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","class":"IncrementalBackupCollection","level":"info","msg":"start uploading files","time":"2026-04-

BackupReturn(error=None, status=<BackupStatus.SUCCESS: 'SUCCESS'>, path='/tmp/weaviate-backup/base_backup01-01-01', backup_id='base_backup01-01-01', size=0, collections=['IncrementalBackupCollection'])

In [8]:
# let's check the backup files
list_backup_files(recursive=True)

['/tmp/weaviate-backup/base_backup01-01-01/backup_config.json',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/backup.json',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-8',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-6',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-1',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-7',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-9',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-11',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-10',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBackupCollection/chunk-2',
 '/tmp/weaviate-backup/base_backup01-01-01/Embedded_at_8079/IncrementalBacku

In [9]:
get_backup_folder_size()

Backup ID                  Size
-------------------------------
base_backup01-01-01    62.91 MB
-------------------------------
TOTAL                  62.91 MB


# Let's add 1k more objects to create some changes

In [10]:
with collection.batch.fixed_size(100) as batch:
    for i in range(10000, 11000):
        # generate 1000 words random content
        random_content = " ".join([f"word{j}" for j in range(1000)])
        batch.add_object(
            {"name": f"Object {i}", "content": random_content},
            vector=np.random.rand(1536).tolist(),
        )

In [11]:
collection.aggregate.over_all().total_count

11000

In [ ]:
# Let's create a second backup, but now referencing our previous base backup
client.backup.create(
    backup_id="incremental_backup01-01-02",
    backend="filesystem", # if you are using s3, or gcs, change it here and provide credentials as environment variables
    incremental_base_backup_id="base_backup01-01-01",
    wait_for_completion=True,
)

{"action":"try_backup","backend":"filesystem","backup_id":"incremental_backup01-01-02","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:45:36-03:00","took":1144125}
{"action":"backup_status","backend":"filesystem","backup_id":"incremental_backup01-01-02","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:45:36-03:00","took":3917}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","hardlinks_supported":true,"level":"info","msg":"backup: probed filesystem hardlink support","time":"2026-04-06T16:45:36-03:00"}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","class":"IncrementalBackupCollection","level":"info","msg":"start uploading files","t

BackupReturn(error=None, status=<BackupStatus.SUCCESS: 'SUCCESS'>, path='/tmp/weaviate-backup/incremental_backup01-01-02', backup_id='incremental_backup01-01-02', size=0, collections=['IncrementalBackupCollection'])

{"action":"hnsw_condensing","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"start hnsw condensing","time":"2026-04-06T16:45:40-03:00"}
{"action":"hnsw_condensing_complete","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"completed hnsw condensing","time":"2026-04-06T16:45:40-03:00"}
{"action":"hnsw_commit_logger_combine_condensed_logs","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","id":"vectors_default","input_first":"/Users/dudanogueira/.local/share/weaviate/incrementalbackupcollection/Bu0eQHd9E617/vectors_default.hnsw.commitlog.d/1775504667.condensed","input_second":"/Users/dudanogueira/.local/share/weaviate/incrementalbackupcollection/Bu0eQHd9E617/vectors_default.hnsw.commitlog.d/1775504688.condensed","level":"info","msg":"successfully combi

In [18]:
get_backup_folder_size()

Backup ID                         Size
--------------------------------------
base_backup01-01-01           62.91 MB
incremental_backup01-01-02     8.39 MB
--------------------------------------
TOTAL                         71.30 MB


In [14]:
client.backup.list_backups(backend="filesystem")

{"action":"list_backup","backend":"filesystem","backup_id":"","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:45:48-03:00","took":2708}


[BackupListReturn(collections=['IncrementalBackupCollection'], status=<BackupStatus.SUCCESS: 'SUCCESS'>, backup_id='incremental_backup01-01-02', started_at=datetime.datetime(2026, 4, 6, 19, 45, 36, 881000, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 4, 6, 19, 45, 38, 884000, tzinfo=TzInfo(0)), size=0.1789334649220109),
 BackupListReturn(collections=['IncrementalBackupCollection'], status=<BackupStatus.SUCCESS: 'SUCCESS'>, backup_id='base_backup01-01-01', started_at=datetime.datetime(2026, 4, 6, 19, 44, 48, 174000, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 4, 6, 19, 45, 0, 177000, tzinfo=TzInfo(0)), size=0.20233411062508821)]

# now let's add 3 thousand new objects

In [15]:
with collection.batch.fixed_size(100) as batch:
    for i in range(11000, 14000):
        # generate 1000 words random content
        random_content = " ".join([f"word{j}" for j in range(1000)])
        batch.add_object(
            {"name": f"Object {i}", "content": random_content},
            vector=np.random.rand(1536).tolist(),
        )

In [16]:
collection.aggregate.over_all().total_count

14000

{"action":"read_disk_use","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"warning","msg":"disk usage currently at 85.30%, threshold set to 80.00%","path":"/Users/dudanogueira/.local/share/weaviate","time":"2026-04-06T16:46:28-03:00"}


# Incremental backups from incremental backups?

Yes! 

Weaviate supports creating chains of incremental backups where each incremental backup can reference any previous backup as its base.

In [19]:
client.backup.create(
    backup_id="incremental_backup01-01-03",
    backend="filesystem", # if you are using s3, or gcs, change it here and provide credentials as environment variables
    incremental_base_backup_id="incremental_backup01-01-02",
    wait_for_completion=True,
)

{"action":"try_backup","backend":"filesystem","backup_id":"incremental_backup01-01-03","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:48:12-03:00","took":1835084}
{"action":"backup_status","backend":"filesystem","backup_id":"incremental_backup01-01-03","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:48:12-03:00","took":4000}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","hardlinks_supported":true,"level":"info","msg":"backup: probed filesystem hardlink support","time":"2026-04-06T16:48:12-03:00"}
{"build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","class":"IncrementalBackupCollection","level":"info","msg":"start uploading files","t

BackupReturn(error=None, status=<BackupStatus.SUCCESS: 'SUCCESS'>, path='/tmp/weaviate-backup/incremental_backup01-01-03', backup_id='incremental_backup01-01-03', size=0, collections=['IncrementalBackupCollection'])

In [20]:
get_backup_folder_size()

Backup ID                         Size
--------------------------------------
base_backup01-01-01           62.91 MB
incremental_backup01-01-02     8.39 MB
incremental_backup01-01-03    36.73 MB
--------------------------------------
TOTAL                        108.04 MB


# Time to restore!

Now, let's delete and restore our data

In [21]:

client.collections.delete_all()
print(client.collections.list_all())

{}


let's retore the entire 14000 objects from our last incremental backup

In [ ]:
client.backup.restore(
    backup_id="incremental_backup01-01-03",
    backend="filesystem", # if you are using s3, or gcs, change it here and provide credentials as environment variables
    wait_for_completion=True,
)

{"action":"try_restore","backend":"filesystem","backup_id":"incremental_backup01-01-03","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:49:21-03:00","took":1130625}
{"action":"restoration_status","backend":"filesystem","backup_id":"incremental_backup01-01-03","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","level":"info","msg":"","time":"2026-04-06T16:49:21-03:00","took":2542}
{"action":"restore","backup_id":"incremental_backup01-01-03","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEAD","build_wv_version":"1.36.6","class":"IncrementalBackupCollection","level":"info","msg":"successfully restored","time":"2026-04-06T16:49:21-03:00"}
{"action":"restore","backup_id":"incremental_backup01-01-03","build_git_commit":"5bbdf290fd","build_go_version":"go1.26.1","build_image_tag":"HEA

BackupReturn(error=None, status=<BackupStatus.SUCCESS: 'SUCCESS'>, path='/tmp/weaviate-backup/incremental_backup01-01-03', backup_id='incremental_backup01-01-03', size=0, collections=['IncrementalBackupCollection'])

In [24]:
collection.aggregate.over_all().total_count

14000

In [ ]:
# one important thing to notice is that you must keep the base backup in order to restore from incrementals.
client.backup.create(
    backup_id="new_base",
    backend="filesystem",
    wait_for_completion=True,
)

In [ ]:
# now I delete the base backup, and try to restore from the incremental, it should fail
import shutil
shutil.rmtree("/tmp/weaviate-backup/base_backup01-01-01")

In [ ]:
client.collections.delete_all()

In [ ]:
get_backup_folder_size()

We have deleted the base backup. So this will fail

In [ ]:
client.backup.restore(
    backup_id="incremental_backup02-01-02",
    backend="filesystem",
    wait_for_completion=True,
)

In [ ]:
collection.aggregate.over_all().total_count

In [ ]:
client.close()